# Target Grouping and Identification

This notebook is intended to plan for Target identification and grouping.
Identification is based on a combination of DICOM Structure Type: 
*RT ROI Interpreted Type (3006,00A4)*
and the Structure ID:
*ROI Name (3006,0026)*

- The RT ROI Interpreted Type values that are defined as Targets are specified.

- Regular expressions are defined that can be used to identify target structures and to group them.

Test examples are Use to demonstrate the functionality of the identification and grouping process.

## Setup

### Imports

In [1]:
from typing import Dict, List, Union

from pathlib import Path
import re

from pprint import pprint

In [2]:
import xml.etree.ElementTree as ET
from itertools import chain


In [3]:
import pandas as pd
import xlwings as xw

In [4]:
from structure_set import StructureSet
from dicom import DicomStructureFile

INFO:metrics.base:Registered calculator: minimum_margins (ContainmentMarginsCalculator)
INFO:metrics.base:Registered calculator: maximum_margin (MaximumMarginCalculator)
INFO:metrics.base:Registered calculator: minimum_distance (MinimumDistanceCalculator)


### Paths

In [ ]:
base_path = Path.cwd()
dicom_path = base_path / "tests"
structure_filter_def = base_path / "src" / "webapp" / "config" / "structure_filter_rules.json"

## Load an example H&N DICOM structure set

### Path to the example DICOM file

In [ ]:
dcm_file_path = dicom_path / "RS.HN_Struct.OROP.dcm"


### Load the DICOM file

In [47]:
dicom_file = DicomStructureFile(top_dir=dicom_path, file_path=dcm_file_path)
pprint(dicom_file.structure_set_info)

INFO:dicom:Successfully loaded DICOM dataset from RS.HN_Struct.OROP.dcm
INFO:dicom:Extracted 3666 contours from 61 ROIs
INFO:dicom:Found 0 frame-of-reference matches and 0 other matches for structure set RS.HN_Struct.OROP.dcm
INFO:dicom:Calculated resolution from structure 'BODY': 0.070 cm/pixel


{'File': WindowsPath("d:/OneDrive - Queen's University/Python/Projects/StructureRelations/tests/RS.HN_Struct.OROP.dcm"),
 'PatientID': 'HN_Struct',
 'PatientLastName': 'Head',
 'PatientName': 'Head^and^Neck^Structures',
 'SeriesDescription': 'ARIA RadOnc Structure Sets',
 'SeriesNumber': '234',
 'StructureSet': 'OROP',
 'StudyID': ''}


### Filter unwanted structures

In [ ]:
filter_report = dicom_file.evaluate_structure_filters(structure_filter_def)
selection = filter_report.SelectedByDefault & filter_report.DisplayedByDefault
meta_data = dicom_file.get_structure_filter_metadata()[selection]
include_structures = list(meta_data['Structure ID'])

In [ ]:
exclude_structures = [
    '[$ZzD].*', 'BODY', 'Avoid.*', '.*Bolus.*', '.*Brain.*', 'Cochlea.*',
    'Esophagus', 'Globe.*', '.*Larynx.*', 'Lens.*', 'Mandible', '.*Optic.*',
    '.*Parotid.*', '.*PRV.*', 'SpinalCanal', 'Submandibular.*', 'Brachial.*',
    'Lip', 'Musc.*', 'Oral.*'
    ]


### Determine the relationships between the structures

In [ ]:
structure_set = StructureSet(dicom_structure_file=dicom_file,
                             include_structures=include_structures)
pprint(list(structure_set.summary().Name))
def get_rt(rel): return rel.relationship_type.label
relations = structure_set.relationship_matrix.map(get_rt)

INFO:structure_set:Building StructureSet from 3666 contour points (unit: cm)
INFO:structure_set:Skipping structure BODY (1) due to filters
INFO:structure_set:Skipping structure Avoid Post (3) due to filters
INFO:structure_set:Skipping structure Brain (4) due to filters
INFO:structure_set:Skipping structure BrainStem (5) due to filters
INFO:structure_set:Skipping structure Cochlea L (6) due to filters
INFO:structure_set:Skipping structure Cochlea R (7) due to filters
INFO:structure_set:Adding structure CTV 56 L (8)
INFO:structure_set:Adding structure CTV 56 R (9)
INFO:structure_set:Adding structure CTV 70 (12)
INFO:structure_set:Skipping structure Esophagus (13) due to filters
INFO:structure_set:Adding structure eval PTV 56 (14)
INFO:structure_set:Adding structure eval PTV 70 (16)
INFO:structure_set:Skipping structure Globe L (17) due to filters
INFO:structure_set:Skipping structure Globe R (18) due to filters
INFO:structure_set:Adding structure GTV (19)
INFO:structure_set:Adding struct

### Utility Functions

In [49]:
def combine_columns(df: pd.DataFrame, columns: List[str], sep=' ')->pd.Series:
    '''Combine text from multiple columns with a separator.

    Args:
        df (pd.DataFrame): The table containing the columns to be merged.
            All of the columns should contain strings or NA values.
        columns (List[str]: The names of the columns to be merged.
        sep (str): A delimiter to place between the text from each column.

    Returns:
        pd.Series: A new text column containing the combined text from each
            column.
    '''
    row_dict = {}
    for index, row in df.iterrows():
        text_items = []
        for col in columns:
            text = row.at[col]
            if text:
                text_items.append(str(text))
        combined_text = sep.join(text_items)
        row_dict[index] = combined_text
    new_col = pd.Series(row_dict)
    return new_col


In [50]:
def to_cgy(text: str)->Dict[str, Union[float, int]]:
    '''Convert numbers with Gy units to cGy and identify fractions.

    Numbers without units or x are assumed to be total dose in cGy.
    Numbers with trailing 'Gy' are converted to cGy.
    Number preceded by x are fractions. dose values are assumed to be dose
    per fraction. Total dose is calculated by:
        $dose_per_fraction x fractions$
    The decimal point may also be represented by a 'p' e.g. 50p4Gy
    If the text does not match any of the valid formats, return the original
    text.

    Args:
        text (str): Dose as a string in one of the following forms:
            ####
            ##Gy
            ##.##Gy
            ##p##Gy
            ####x#
            ##Gyx#
            ##.##Gyx#
            ##p##Gyx#

    Returns:
        Tuple[float, int]: _description_
    '''
    if not text:
        dose_dict = {'TotalDose': text, 'Fractions': None}
        return pd.Series(dose_dict)
    # Convert 'p' to decimal point.
    try:
        text_cnv1 = text.replace('p', '.')
    except AttributeError:
        dose_dict = {'TotalDose': text, 'Fractions': None}
        return pd.Series(dose_dict)
    # Find fractions
    dose_parts = text_cnv1.split('x', 1)
    if len(dose_parts) > 1:
        try:
            fractions = int(dose_parts[1])
        except ValueError:
            dose_dict = {'TotalDose': text, 'Fractions': None}
            return pd.Series(dose_dict)
    else:
        fractions = None
    dose_str = dose_parts[0]
    # Convert Gy to cGy
    if dose_str.endswith('Gy'):
        try:
            dose = float(dose_str[:-2])  # Drop the Gy suffix
        except ValueError:
            dose_dict = {'TotalDose': text, 'Fractions': None}
            return pd.Series(dose_dict)
        dose = dose * 100   # Gy to cGy conversion
    else:
        try:
            dose = float(dose_str)
        except ValueError:
            dose_dict = {'TotalDose': text, 'Fractions': None}
            return pd.Series(dose_dict)
    # Convert dose per fraction to total dose
    if fractions:
        total_dose = dose * fractions
    else:
        total_dose = dose
    dose_dict = {'TotalDose': total_dose, 'Fractions': fractions}
    return pd.Series(dose_dict)


In [51]:
def extract_name_group(names: pd.DataFrame, re_pattern: re.Pattern,
                       match_column: str, idx: pd.Series = None)->pd.DataFrame:
    '''Extract portions of a structure name.

    The re_pattern is applied to the 'Remainder' column to extract named parts.
    The resulting DataFrame is merged with names and the Remainder columns is
    updated with new Remainders from the extraction.

    Args:
        nt_names (pd.DataFrame): A table with structure names. It must contain
            a column 'Remainder', which is used as the starting point for
            extracting name parts.
        re_pattern (re.Pattern): A regular expression with named groups.  It
            must contain one named group that will always be present if a
            successful match is made.  It must also contain a 'Remainder'
            named group that contains the unmatched part of the structure name.
        match_column (str): The name of the named group that is always present
            when a successful match is made.  Used to update the 'Remainder'
            column.
        idx (pd.Series, optional): A mask type index to a subset of names to
            apply the match to.  If not supplied, all names are matched.
            Default is None.

    Returns:
        pd.DataFrame: The supplied table with new columns containing the
            structure name parts.
    '''
    # Extract group parts based on regular expression.
    if idx is not None:
        extr_names = names.loc[idx, 'Remainder'].str.extract(re_pattern)
    else:
        extr_names = names.loc[:, 'Remainder'].str.extract(re_pattern)

    # Merge extracted group parts with structure names.
    names = names.merge(extr_names, how='left',
                            left_index=True, right_index=True,
                            suffixes=('', '_ex'))

    # Update Remainder text
    nt_idx = names[match_column].isna()
    # Where a match was not found, keep the original Remainder text, otherwise
    # update Remainder with resulting Remainder after the match.
    names.Remainder = names.Remainder.where(nt_idx, names.Remainder_ex)
    names.drop(columns=['Remainder_ex'], inplace=True)
    return names


- Structures that are not used for dose evaluation (e.g., optimization 
  structures, high/low dose regions) should be prefixed with a 
  <u>z</u> or <u>_</u> character.

### Non-Evaluated Nomenclature

- Structures that are not used for dose evaluation (e.g., optimization 
  structures, high/low dose regions) should be prefixed with a 
  <u>z</u> or <u>_</u> character.
- Text after the 'z' or '_' can be anything and is not evaluated.

In [56]:
not_evaluated_pat = ''.join([
    r'(?P<NotEvalChar>',   # Start of named group NotEvalChar
    r'[zZ_]',              # Z or _ character
    r')',                  # End of named group NotEvalChar
    r'(?P<NotEvaluated>',  # Start of named group NotEvaluated
    r'.*',                 # Remainder of the text.
    r')',                  # End of named group NotEvaluated
    ])


### Target Nomenclature

In [58]:
group_definitions = {}

#### Target Type

- The first set of characters must be one of the allowed target types:
    - GTV
    - CTV
    - ITV
    - IGTV (Internal Gross Target Volume—gross disease with margin for motion)
    - ICTV (Internal Clinical Target Volume—clinical disease with margin for motion)
    - PTV
    - PTV! for low-dose PTV volumes that exclude overlapping high-dose volumes

- Additional Target types not defined in TG-263:
  - Nodes, Node:  Used to identify a nodal volume that is not a GTV, but used to define a CTV or   PTV.
  - Iliac Vessels Special case of pelvic vessels used as a surrogate for pelvic nodes.
  - Edema: Used as a contrasted-based indicator of CTV volumes in CNS tumours.
  - Cavity: Used in breast cancer, where the GTV is defined as the surgical cavity.
  - Operative Bed: Used for post-surgery cases where the target is the operative bed.
  - HTV: High-risk target volume.
  - HRV: High-risk volume.

In [ ]:
target_type_def = {
    'GTV': 'Gross Target Volume',
    'CTV': 'Clinical Target Volume',
    'PTV': 'Planning Target Volume',
    'ITV': 'Internal Target Volume',
    'IGTV': 'Internal Gross Target Volume',
    'ICTV': 'Internal Clinical Target Volume',
    'PTV!': 'Partial Planning Target Volume',
    # Additional local target types not defined in TG-263:
    'Nodes': 'Nodal Volume',
    'Node': 'Nodal Volume',
    'Iliac Vessels': 'Nodal Volume',
    'Edema': 'CTV Surrogate',
    'Cavity': 'GTV Surrogate',
    'Operative Bed': 'GTV Surrogate',
    'HTV': 'Clinical Target Volume',
    'HRV': 'Clinical Target Volume',
    }

# Create a regular expression pattern to match the target types defined in target_type_def.
target_type = '|'.join(target_type_def.keys())
target_type_pat = ''.join(
    r'(?P<TargetType>'  # Start of required *TargetType* group
    f'{target_type}'    # contains '|' separated items from target_type.
    r')'                # End of *TargetType* group.
    )

# Add the target type definitions to the group_definitions dictionary.
group_definitions['TargetType'] = target_type_def

### Target Modifier Prefixes
- `eval_` Target volume explicitly intended for DVH evaluation
- `opt_` Target volume only intended for optimization. (TG263 recommends that 
  such structures should be prefixed with a `z` or an `_`, but this may be too 
  generic of an indicator.)


In [ ]:
# MOD (A prefix identifying *eval* or *opt* targets)
target_group_options['mod_types'] = ['eval', 'opt', 'mod']
target_pattern_list.append(
    r'(?P<Mod>'     # Start of *Mod* group
    r'{mod_types}'  # contains '|' separated items from mod_types.
    r')?'           # End of optional *Mod* group
    r'[ _]*'        # Optional space or '_'
    )

#### Target Classifier

- If used, the target classifier is placed after the target type with no spaces.
    - Allowed target classifiers are listed below: 
    - n: nodal (e.g., PTVn) 
    - p: primary (e.g., GTVp) 
    - sb: surgical bed (e.g., CTVsb) 
    - par: parenchyma (e.g., GTVpar) 
    - v:venous thrombosis (e.g., CTVv) 
    - vas: vascular (e.g., CTVvas)

In [73]:
target_classifier_def = {
    'n': 'nodal',
    'p': 'primary',
    'sb': 'surgical bed',
    'par': 'parenchyma',
    'v': 'venous thrombosis',
    'vas': 'vascular'
    }


In [74]:
target_classifier_pat = ''.join([
    r'(',               # Start of optional group
    r'par|vas|sb|n|p|v',  # Target classifier options
    r')?'               # End of the optional group
    ])


#### Target Number

- For multiple spatially distinct targets Arabic numerals are used after the 
  target type and classifier (e.g., PTV1, PTV2, GTVp1, GTVp2).

In [75]:
target_number_pat = ''.join([
    r'(',         # Start of optional group
    r'[0-9]{1,2}',  # Target number as 1 or 2 digits
    r')?'         # End of the optional group
    ])


#### Base Target
- Base target include first three parts of target structure name:
  1. Target Type
  2. Target Classifier
  3. Target Number

- The base target is grouped separately because it can be used as a cropping 
  designator for OARs. e.g. `Brain-GTV`

In [76]:
base_target_pat = ''.join([
    r'(?P<BaseTarget>',      # Start of named group BaseTarget
    r'(?P<TargetType>',        # Start of named group TargetType
    target_type_pat,             # Target Type pattern definition
    r')',                      # End of the TargetType group
    r'(?P<TargetClassifier>',  # Start of optional named group TargetClassifier
    target_classifier_pat,       # Target Classifier pattern definition
    r')?',                     # End of the optional TargetClassifier group
    r'(?P<TargetNumber>',      # Start of optional named group TargetNumber
    target_number_pat,           # Target Number pattern definition
    r')?',                     # End of the optional TargetNumber group
    r')',                    # End of the BaseTarget group
    ])


- Imaging modality follows the type/classifier/enumerator with an underscore 
  and then the image modality type (CT, PT, MR, SP)

- Image sequence order is indicated by a number immediately following the 
  image modality.

- Multiple modalities can be included.  No additional underscore is used.

In [77]:
modality_def = {
    'CT': 'CT',
    'PT': 'Pet',
    'MR': 'MRI',
    'US': 'Ultrasound',
    'SP': 'Spect'
    }


In [78]:
modality_pat = ''.join([
    r'(',            # Start of optional group
    r'(?:_)',          # Underscore delimiter (Not captured)
    r'(?P<Modality>',  # Start of optional named group Modality
    r'('                 # Beginning of optional repeat group
    r'(CT|PT|MR|US|SP)',   # Modality designator group
    r'[0-9]{0,2}',         # Optional sequence number as 1 or 2 digits
    r'){1,2}'            # repeatable group for multiple modalities
    r')'               # End of named group Modality
    r')?'            # End of optional group
    ])


#### Structure Indicators
- Structure indicators follow the type/classifier/enumerator/imaging with an underscore prefix
- Structure indicators are values from the approved structure nomenclature list.
- Examples: CTV_A_Aorta, CTV_A_Celiac, GTV_Preop, PTV_Boost, PTV_Eval, PTV_MR2_Prostate


**Note:**
- Relative dose indicators have a similar pattern to Structure Indicators, but 
  are limited to three text strings: 'High', 'Mid', or 'Low' (see next section).
- None of the current valid Structure Indicators begin with this text.
- Add a check to the pattern for these three text strings.

In [79]:
struct_ind_pat = ''.join([
    r'(',                      # Start of optional group
    r'(?:_)',                    # Underscore delimiter (Not captured)
    r'(?!Hig|Mid|Low)',          # Exclude text that begins with one of these patterns

    r'(?P<StructureIndicator>',  # Start of named group StructureIndicator
    r'(',                          # Start of group
    basic_sub_structure_pat,         # Sub-structure pattern definition
    r')+'                          # End of repeatable group
    r')',                        # End of named group StructureIndicator
    r')?'                      # End of optional group
    ])


- If the structure is cropped back from the external contour for the patient, 
  then the quantity of cropping by “-xx” millimeters is placed at the end of 
  the target string. The cropping length follows the dose indicator, with the 
  amount of cropping indicated by xx millimeters 
  (e.g., PTV_Eval_7000-08, PTV-03, CTVp2-05).

In [80]:
target_crop_pat = ''.join([
    r'(?P<ExternalCrop>',  # Start of optional named group ExternalCrop
    r'(?P<Sign>-)',          # Negative sign '-' as its own named group
    r'(?P<Size>[0-9]{2})',   # 2-digit Number
    r')?'                  # End of optional group
    ])


- If a custom qualifier string is used, the custom qualifier is placed at the 
  end after a ‘^’ character (e.g., PTV^Physician1, GTV_Liver^ICG).
- Include *custom_qualifier_pat* from non-target structures

#### Dose Specifier

- Dose specifier is placed at the end of the target string prefixed with an underscore character.

- Dose specifier can be one of:
   - Relative Dose Level
   - Numeric dose
   - Dose per Fraction and number of Fractions


##### Relative Dose
- Relative dose is recommended
    - High (e.g., PTV_High, CTV_High, GTV_High)
    - Mid: (e.g., PTV_Mid, CTV_Mid, GTV_Mid)
    - Low (e.g., PTV_Low, CTV_Low, GTV_Low) ◦ 

- Mid+2-digit enumerator: allows specification of more than three relative 
  dose levels (e.g., PTV_Low, PTV_Mid01, PTV_Mid02, PTV_Mid03, PTV_High). 
- Lower numbers correspond to lower dose values.

In [81]:
rel_dose_pat = ''.join([
    r'(?P<RelativeDose>',     # Start of named group RelativeDose
    r'High|',                   # 'High' relative dose    OR
    r'Low|',                    # 'Low'  relative dose    OR
    r'Mid',                     # 'Mid'  relative dose   with
    r'(?P<RelativeDoseLevel>',  # Optional RelativeDoseLevel group
    r'[0-9]{2}',                  # 2-digit number
    r')?',                      # End of Optional RelativeDoseLevel group
    r')'                      # End of RelativeDose group
    ])


##### Numeric Dose
- Units of cGy are recommended for numeric dose values. (e.g., PTV_5040).
- When specified in units of Gy, then ‘Gy’ should be appended to the numeric 
  value of the dose (e.g., PTV_50.4Gy). 
- For systems that do not allow use of a period, the ‘p’ character should be 
  substituted (e.g., PTV_50p4Gy)

In [82]:
numeric_dose_pat = ''.join([
    r'(?P<NumericDose>',  # Start of optional named group NumericDose
    r'[0-9]+',              # Number before decimal place
    r'[.p]?',               # '.' or 'p' as optional decimal place
    r'[0-9]*',              # Optional Number after decimal place
    r'[Gy]*',               # Optional units of Gy
    r')'                  # End of NumericDose group
    ])


##### Dose per Fraction and number of Fractions
- If the dose indicated must reflect the number of fractions used to reach the 
  total dose, then the numeric values of <u>dose per fraction</u> in cGy, or 
  in Gy with the <u>unit specifier</u>, '<u>x</u>' followed by the <u>number 
  of fractions</u> (e.g., PTV_Liver_2000x3 or PTV_Liver_20Gyx3).

In [83]:
dose_fraction_pat = ''.join([
    r'(?P<DoseFractionation>',  # Start of  named group DoseFractionation
    r'[0-9]+',                    # Number before decimal place
    r'[.p]?',                     # '.' or 'p' as optional decimal place
    r'[0-9]*',                    # Optional Number after decimal place
    r'[Gy]*',                     # Optional units of Gy
    r'x',                         # Fractions delimiter 'x'
    r'[0-9]+',                    # Number of fractions
    r')'                        # End of DoseFractionation group
    ])


- Dose Fractionation must be specified before Numeric Dose because otherwise Numeric Dose will catch part of Dose Fractionation

In [84]:
dose_specifier = ''.join([
   '(',                   # Beginning of optional Dose Specifier group
    r'(?:_)',               # Underscore delimiter (Not captured)
    r'(?P<DoseSpecifier>',  # Start of named group DoseSpecifier
    rel_dose_pat,             # Relative Dose pattern definition
    '|',                    # OR
    dose_fraction_pat,        # Dose Fractionation pattern definition
    '|',                    # OR
    numeric_dose_pat,         # Numeric Dose pattern definition
    ')'                     # End of Dose Specifier options
    ')?',                 # End of optional Dose Specifier group
   ])


#### Custom Qualifier
- Custom Qualifier indicated by '^' e.g. Lungs^Ex

In [85]:
target_custom_qualifier_pat = ''.join([
    r'('                 # Start of optional group
    r'(?:\^)'              # '^' character (not captured)
    r'(?P<CustomTarget>',  # Start of optional named group Custom
    r'.+'                    # Remainder of text
    r')'                   # End of named group Custom
    r')?'                # End of optional group
    ])


#### Combine Target patterns
**Order of Name Components:**  (Underline indicates required component)

1. <u>Base Target</u>  (consists of three parts)
   1. <u>Target Type</u>
   2. Target Classifier
   3. Target Number
2. Modality
3. Structure Indicator
4. Dose Specifier
5. Target Cropping from External
6.  Custom Qualifier

- Pattern must match the entire string


In [86]:
target_pattern = ''.join([
    r'(',                         # Start of all target group patterns
    base_target_pat,              # Base Target pattern definition
    modality_pat,                 # Target Modality pattern definition
    struct_ind_pat,               # Structure pattern definition
    dose_specifier,               # Dose Specifier pattern definition
    target_crop_pat,              # Target Cropping pattern definition
    target_custom_qualifier_pat,  # Custom Qualifier pattern definition
    r')',                         # End of all target group patterns
    ])


In [ ]:
structure_pat = re.compile(target_pattern)


In [92]:
examples_file = reference_path / 'examples.txt'
examples = examples_file.read_text().splitlines()

In [93]:
matched_structure_list = []
for structure in examples:
    mtch = structure_pat.fullmatch(structure)
    if mtch:
        mtch_dict = mtch.groupdict()
        mtch_dict['Structure'] = structure
    else:
        mtch_dict = {'Structure': structure}
    matched_structure_list.append(mtch_dict)

matched_structures = pd.DataFrame(matched_structure_list)
matched_structures.drop_duplicates(inplace=True)
matched_structures.set_index('Structure', inplace=True)


### Get Dose Values

In [94]:
dose_values = matched_structures.DoseSpecifier.apply(to_cgy)
matched_structures = matched_structures.join(dose_values)


### Identify match failures

In [95]:
unmatched_idx = matched_structures.isna().all(axis='columns')
unmatched_idx.name = 'Unmatched'
matched_structures = matched_structures.join(unmatched_idx)


### Derived Structures

Derived Structures are discussed in the beginning of the TG263 document 
(see below), but no guidance is explicitly given on nomenclatures for 
derivative structures except for targets cropped back from skin 
(e.g. `PTV-03`) and low-dose PTV volumes that exclude overlapping 
high-dose volumes (`PTV!`).

> **4.4 Derived and Planning Structures**<br>
> Derivative structures are formed from target or non-target structures, 
> typically using Boolean operations, e.g., intersection (x AND y), combination 
> (x OR y), subtraction (x AND NOT y), and margins (x+1.0). Five institutions 
> indicated that nomenclatures for derivative structures were used to define
> conditions for evaluating the dose distribution (e.g., OAR contour excluding 
> PTV). Variations in several structures were common (e.g., body-ptv, PTV_EVAL,
>  eval_PTV), but wide variation was noted for structures involving multiple 
>  concepts (e.g., NS_Brain-PTVs, optCTV-N2R5L_MRT1_ex-3600-v12).

- Parsing instructions have been included for targets cropped from OARs, since 
  this form appears in multiple TG263 examples.

- Derived *Normal Tissue* examples are discussed below.


#### *Normal Tissue* Structures
**The following 'Normal Tissue' structure names are given in the examples, but not discussed in the text.**

- `E-PTV_Ev05_xxxx` represents all tissue excluding the 5 mm expanded PTV. 
  Generated by subtracting the 5 mm expanded PTV receiving a dose of xxxx cGy 
  from the external contour.
  
- `E-PTV_xxxx` represents all tissue excluding the PTV. Generated by 
  subtracting the PTV receiving a dose of xxxx cGy from the external contour.

- Should this be a special case structure pattern?
- Below is a regex pattern based on the the above examples

In [59]:
normal_t_pat = ''.join([
    r'^'                  # Beginning of string.
    r'(?P<NormalTissue>', # Start of named group NormalTissue
    r'E-'                   # The text 'E-'
    r'(?P<TargetCrop>',     # Start of named group TargetCrop
    r'PTV'                    # The text 'PTV'
    r'(?:'                    # Start of optional non-capturing group
    r'_Ev'                      # the text '_Ev'
    r'(?P<TargetExpansion>',    # Start of named group TargetExpansion
    r'[0-9]{2}'                   # Expansion size as 2 digits
    r')'                        # End of TargetExpansion group
    r')?'                     # End of optional group
    r'(?:'                    # Start of optional non-capturing group
    r'_'                        # '_' s delimiter
    r'(?P<TargetDose>',         # Start of named group TargetDose
    r'[0-9]{4}'                   # Target dose in cGy using 4 digits
    r')'                        # End of TargetDose group
    r')?'                     # End of optional non-capturing group
    r')'                    # End of TargetCrop group
    r')',                 # End of NormalTissue group
    r'$'                  # End of string.
    ])


### Target Motion Modifiers
No mention is made of 4DCT related target designations.  
The following motion related designators may be useful.
- `_MIP`
- `_AVE`
- `_##%` (Where `##` is the breathing phase the target was contoured on.)

### Target Organ Subtraction
There are times when modified targets are generated by subtracting a critical 
OAR from the initial target volume
- `-<Organ>`  (Where `<Organ>` represents and valid OAR structure name)

### Targets Combined
- `_Total` as the combined structure for all targets at the same dose level.

### Additional Target Type
- HTV as High Risk target Volume

### Target Expansion
PTVs are sometimes expanded to evaluate the conformality of a plan.
While not mentioned in the text, the supplied TG263 examples suggest the 
following format:
- `_Ev##` (where `##` is the uniform expansion size in mm)


### Target Subgroup
For optimization purposes a target volume may be divided into subsections 
based on the proximity to hight dose targets.  The TG263 document implies that 
such structures should be prefixed with a `z` or an `_`.

An alternative would be to append a letter suffix to the full target volume to 
designate that it is a subsection.
- `~a`, `~b` `~c`

The `~` delimiter would have the same meaning of *partial structure* that it 
has for OAR structures.


### Additional Target Classifiers
- `_Edema` as target volume based on CNS edema imaging.
- `_Cavity` as target volume based on a surgical cavity.
- `_PREOP` as target volume based on pre-operative imaging.
- `_RES` as target volume for post-op residual disease


### OAR Multiple Occurrence. 
A suffix noting the occurrence number of the structure.
- `<Organ>_##` (Where `<Organ>` represents and valid OAR structure name 
  and `##` represents the occurrence number)


### Bolus Types
The term *Bolus* may be too generic. 
 Alternative bolus terms may also be needed.
- `BartsBolus`
- `WetGauze`
- `PinkWax`

### Foreign Location Reference Objects
- `BB`
- `Fiducial`
- `MARKER`
- `OVOID`
- `Wire`


### Structures for Field Placement Reference
- `MatchPlane`
- `Baseline`
- `FieldEdge`


### Structures for Correcting Density
- `Air`
- `Contrast`

### Foreign Objects as Structures
- `CIED`
- `Prosthesis`
- `Implant`
- `Expander`
- `Screws`
- `Staples`
- `Hardware`
- `Metal`
- `Dental`
- `Anastomosis`
- `Stoma`

**Structure Set ROI Sequence (3006,0020)**

- **ROI Number (3006,0022)**
    Identification number of the ROI. The value of *ROI Number (3006,0022)* 
    shall be unique within the Structure Set in which it is created.

- **ROI Name (3006,0026)**
    User-defined name for ROI.


**RT ROI Observations Sequence (3006,0080)**

- **Referenced ROI Number (3006,0084)**
    Uniquely identifies the referenced ROI described in the 
    *Structure Set ROI Sequence (3006,0020)*.

- **ROI Observation Label (3006,0085)**
    User-defined label for ROI Observation.

- **RT ROI Interpreted Type (3006,00A4)**
    Type of ROI.
  - Defined Terms:

|Term|Definition|
|----|----------|
|EXTERNAL|external patient contour|
|PTV|Planning Target Volume (as defined in ICRU50)|
|CTV|Clinical Target Volume (as defined in ICRU50)|
|GTV|Gross Tumor Volume (as defined in ICRU50)|
|TREATED_VOLUME|Treated Volume (as defined in ICRU50)|
|IRRAD_VOLUME|Irradiated Volume (as defined in ICRU50)|
|BOLUS|patient bolus to be used for external beam therapy|
|AVOIDANCE|region in which dose is to be minimized|
|ORGAN|patient organ|
|MARKER|patient marker or marker on a localizer|
|REGISTRATION|registration ROI|
|ISOCENTER|treatment isocenter to be used for external beam therapy|
|CONTRAST_AGENT|volume into which a contrast agent has been injected|
|CAVITY|patient anatomical cavity|
|BRACHY_CHANNEL|brachytherapy channel|
|BRACHY_ACCESSORY|brachytherapy accessory device|
|BRACHY_SRC_APP|brachytherapy source applicator|
|BRACHY_CHNL_SHLD|brachytherapy channel shield|
|SUPPORT|external patient support device|
|FIXATION|external patient fixation or immobilization device|
|DOSE_REGION|ROI to be used as a dose reference|
|CONTROL|ROI to be used in control of dose optimization and calculation|